# DIT5411 — Train one model per Colab session

**Runtime → Change runtime type → T4 GPU.**

Each run trains **one model** for 2 short epochs (5,000 steps/epoch ≈ 1.5h total), which fits a free Colab session. Run this notebook once per model: set `MODEL` to `A`, `B`, or `C` in the training cell, then run all cells. Each run downloads its `.keras` model and prints accuracy.

Accuracy is lower than a full 20,394-step run, but all models use the **same** settings, so the comparison is valid for the assignment.

In [ ]:
import os, sys, subprocess, glob, json, shutil

if not os.path.isfile('train.py'):
    if not os.path.isdir('DIT5411-LuiYuHin'):
        subprocess.run(['git', 'clone', '-q', 'https://github.com/cveg1234/DIT5411-LuiYuHin.git'], check=True)
    os.chdir('DIT5411-LuiYuHin')
subprocess.run(['git', 'pull', '-q'])
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])

OUT_DIR = 'results'
os.makedirs(OUT_DIR, exist_ok=True)
import tensorflow as tf
print('Working dir:', os.getcwd())
print('GPU:', tf.config.list_physical_devices('GPU'))

## Dataset (one time per runtime)

In [ ]:
DATA_DIR = os.path.join(os.getcwd(), 'data', 'cleaned_data')
if not os.path.isdir(DATA_DIR) or len(os.listdir(DATA_DIR)) < 1000:
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/chenkenanalytic/handwritting_data_all.git'], check=True)
    with open('all_data.zip', 'wb') as out:
        for part in sorted(glob.glob('handwritting_data_all/all_data.zip.*')):
            with open(part, 'rb') as f:
                out.write(f.read())
    subprocess.run([sys.executable, 'extract_dataset.py',
                    '--zip', 'all_data.zip', '--dest', DATA_DIR], check=True)
print('Character folders:', len(os.listdir(DATA_DIR)))

## Train ONE model (change MODEL to A / B / C)
Fast config: 128px input (full detail), 5,000 steps × 2 epochs, validation capped at 200 steps. Do NOT lower resolution — 64px blurs the characters and the model cannot learn. Same settings for every model so the comparison stays valid. ~1h per model.

In [ ]:
MODEL = 'B'   # <-- change to 'A', 'B' or 'C' and run this cell
r = subprocess.run([sys.executable, 'train.py', '--data-dir', DATA_DIR,
                    '--output-dir', OUT_DIR, '--epochs', '2',
                    '--steps-per-epoch', '5000',
                    '--validation-steps', '200',
                                        '--models', MODEL, '--resume'])
print(f'Model {MODEL} exit code:', r.returncode)
print('Model file:', os.path.isfile(os.path.join(OUT_DIR, f'model_{MODEL}.keras')))

## Results

In [ ]:
from IPython.display import Markdown, display
rp = os.path.join(OUT_DIR, 'RESULTS.md')
display(Markdown(open(rp, encoding='utf-8').read()) if os.path.isfile(rp)
        else 'RESULTS.md not generated yet — finish training first.')

## Download the trained model

In [ ]:
from google.colab import files
src = os.path.join(OUT_DIR, f'model_{MODEL}.keras')
ckpt = os.path.join(OUT_DIR, 'checkpoints', f'model_{MODEL}_latest.keras')
path = src if os.path.isfile(src) else (ckpt if os.path.isfile(ckpt) else None)
if path is None:
    print('No model file yet — finish training first.')
else:
    print('Downloading', path, '(', round(os.path.getsize(path)/1024/1024), 'MB )')
    files.download(path)